# 11.14 看見物件，和看懂整張圖，有什麼距離？


一張照片裡有「人、杯子、桌子」，還不足以說明發生什麼事。那個人是在端起杯子，還是杯子只是放在桌上？杯子在誰的左邊？描述整張圖，要把物件、動作與彼此關係一起說清楚。這也不表示模型知道照片之前或之後發生的故事；沒有拍到的部分仍可能不知道。

前置是[11.4圖片問答](https://birdhackor.github.io/tiny-perceptron-vlm/11.4.html)與[10.2圖片序列](https://birdhackor.github.io/tiny-perceptron-vlm/10.2.html)。先用最小的左右例子：紅條在藍條左邊與右邊，兩張圖的顏色總量完全相同。若先把同一小區的像素平均，這個差別就可能消失。像把兩張座位表攪成「各有一個紅色座位、一個藍色座位」，再聰明的人也不能從這份摘要還原誰坐哪邊。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/practical_order.svg")))

In [ ]:
from tiny_perceptron.natural_concepts import picture_order_report

report = picture_order_report()
print("不同的像素數", report["different_pixels"])
print("4乘4平均摘要相同", report["same_4_by_4_summary"])
print("完整像素序列相同", report["same_pixel_sequence"])

程式建立兩張32乘32的合成圖片，只交换第一個8像素寬區域裡的紅、藍順序。輸出是128、True、False。不同像素數是把RGB各通道的數字也分開數；不是128個彩色位置。4乘4摘要的每格平均8乘8原始像素，所以這次交換恰好留在同一格內，平均完全相同。這只是輸入會丟資訊的反例，沒有載模型，也沒有證明保留序列便能理解自然照片。

自然照片還有光線、遮擋、姿勢與多個物件。需要含整圖描述或關係問答的示範，才能教到「拿著」「左邊」「正在做什麼」；只增加顏色標籤不會自動教會這些關係。[20.9](https://birdhackor.github.io/tiny-perceptron-vlm/20.9.html)介紹自然照片的逐項判準，實際場景、動作與關係題成績見[20.13](https://birdhackor.github.io/tiny-perceptron-vlm/20.13.html)。有些系統先用長方形框標出人、杯子等物件的位置，這叫物件偵測；框的位置可以提供線索。整圖問答也可以直接接收圖片區塊的序列，學習利用其中的物件與位置關係。

練習在紙上畫兩個8乘8平均格：原本紅、藍都在第一格，接著把紅條移到第二格。想一想各格的顏色平均是否仍相同。再為「有人拿著杯子」寫一題需要觀察整張照片的問題，以及一個照片不能支持的猜測。
